# 🔬 Cervical Cancer Stage Classification — Kaggle Training Notebook
# Backbone + Mamba v1 · 5-Class Classification · ConvNeXt V2 / EfficientNetV2-L / Swin V2 / MaxViT

**Model:** Strong pretrained image backbone + Mamba-based classifier  
**Classes:** `Normal → CIN1 → CIN2 → CIN3 → Cancer`  
**Loss:** Class-balanced Focal Loss  
**Augmentation:** CLAHE · Rotation · Flip · Color Jitter · Brightness/Contrast · Blur · MixUp · CutMix  
**Tricks:** AMP · Gradient accumulation · Cosine LR schedule · Warmup · Early stopping · Best-F1 checkpoint

**Backbone options:** ConvNeXt V2 Large, EfficientNetV2-L, Swin Transformer V2, MaxViT  
**Goal:** maximize validation macro F1-score on cervical cancer image classification

---
> ⚙️ **Runtime:** Set to *GPU* (Notebook settings → Accelerator → GPU) before running.


## Step 0 — Environment & GPU Check

In [ ]:
# ── Install and verify dependencies for Kaggle ───────────────────────────────
import subprocess
import sys
import torch
from pathlib import Path

IS_KAGGLE = Path('/kaggle').exists()
if not IS_KAGGLE:
    raise RuntimeError('This notebook is configured for Kaggle only. Please run it in a Kaggle notebook.')

WORK_BASE = Path('/kaggle/working')
REPO_DIR = WORK_BASE / 'repo'
BACKEND_DIR = REPO_DIR / 'backend'
REQ_FILE = BACKEND_DIR / 'requirements.txt'

if BACKEND_DIR.exists() and REQ_FILE.exists():
    print(f'Installing dependencies from {REQ_FILE}...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ_FILE)],
        capture_output=True,
        text=True,
        timeout=300
    )
    if result.returncode != 0:
        print(f'⚠️  Pip install had warnings/errors (code {result.returncode}):')
        if result.stderr:
            print(result.stderr[:500])  # Print first 500 chars of error
    else:
        print('✅ Dependencies installed.')
else:
    if not BACKEND_DIR.exists():
        print(f'⚠️  Backend directory not found: {BACKEND_DIR}')
        print('   Make sure to run the clone cell first.')
    if not REQ_FILE.exists():
        print(f'⚠️  requirements.txt not found: {REQ_FILE}')

print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU             : {props.name}")
    print(f"VRAM            : {props.total_memory / 1024**3:.1f} GB")
    print(f"CUDA version    : {torch.version.cuda}")
    try
        print(f"cuDNN version   : {torch.backends.cudnn.version()}")
    except RuntimeError as e:
        print('⚠️  cuDNN version check failed:')
        print(f'   {e}')
        print('   This often means the running cuDNN library differs from the bundled PyTorch cuDNN.')
        print('   You can still continue if CUDA is available and no other runtime errors appear.')
else:
    print('⚠️  No GPU detected — training will be slow.')
    print('   Notebook settings → Accelerator → GPU')

print('\n✅ Environment check complete!')

## Step 1 — Clone Repository

In [ ]:
import subprocess
import shutil
from pathlib import Path

IS_KAGGLE = Path('/kaggle').exists()
if not IS_KAGGLE:
    raise RuntimeError('This notebook is configured for Kaggle only. Please run it in a Kaggle notebook.')

WORK_BASE = Path('/kaggle/working')
WORK_BASE.mkdir(parents=True, exist_ok=True)

REPO_URL = 'https://github.com/Shubh-Rawat7/Cervical-Cancer-Classifier.git'
REPO_DIR = WORK_BASE / 'repo'

print('Environment : Kaggle')
print(f'Working dir : {WORK_BASE}')
print(f'Repo URL    : {REPO_URL}')

if REPO_DIR.exists():
    print(f'Removing existing repo at {REPO_DIR} to avoid stale files...')
    shutil.rmtree(REPO_DIR)

print(f'\n⏳ Cloning repository...')
r = subprocess.run(
    ['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)],
    capture_output=True,
    text=True,
    timeout=120
)

if r.returncode != 0:
    print(f'❌ Clone failed with code {r.returncode}')
    print(f'STDERR: {r.stderr}')
    raise RuntimeError(f'Git clone failed:\n{r.stderr}')

print(r.stdout.strip())

# Verify the clone succeeded
if not REPO_DIR.exists():
    raise RuntimeError(f'Clone completed but directory not found at {REPO_DIR}')

print('\nRepo contents:')
try:
    items = sorted(REPO_DIR.iterdir())
    for p in items[:15]:
        marker = '/' if p.is_dir() else ''
        print(f'  {p.name}{marker}')
    if len(items) > 15:
        print(f'  ... and {len(items) - 15} more items')
except Exception as e:
    print(f'  (Could not list contents: {e})')

# Verify backend/scripts/train_hybrid.py exists (the actual training script)
train_py = REPO_DIR / 'backend' / 'scripts' / 'train_hybrid.py'
if train_py.exists():
    print(f'✅ backend/scripts/train_hybrid.py found ({train_py.stat().st_size} bytes)')
else:
    print(f'⚠️  backend/scripts/train_hybrid.py not found at {train_py}')
    print('   (repo may still be incomplete)')

print(f"\n✅ Repo ready at: {REPO_DIR}")

In [ ]:
# ── Verify the training script for Kaggle ─────────────────────────────────────
from pathlib import Path
import sys
import time

WORK_BASE = Path('/kaggle/working')
REPO_DIR = WORK_BASE / 'repo'

# Wait briefly for the repo to be ready if cloning just completed
for attempt in range(3):
    if REPO_DIR.exists():
        break
    if attempt < 2:
        print(f'⏳ Waiting for repo to be ready (attempt {attempt+1}/3)...')
        time.sleep(1)

if not REPO_DIR.exists():
    print(f'❌ Repo directory not found at {REPO_DIR}')
    print('⚠️  Make sure you ran the clone cell (Step 1) successfully.')
    print('   The cell should complete with "✅ Repo ready at: /kaggle/working/repo"')
    raise FileNotFoundError(f'Repository not found at {REPO_DIR}. Run the clone cell first.')

# Look for the actual training script: prefer backend/train.py on Kaggle
candidates = [
    REPO_DIR / 'backend' / 'train.py',
    REPO_DIR / 'backend' / 'scripts' / 'train_hybrid.py',
    REPO_DIR / 'train.py',
]

TRAIN_SCRIPT = None
for p in candidates:
    if p.exists():
        TRAIN_SCRIPT = p
        print(f'✅ Found training script: {p.relative_to(REPO_DIR)}')
        break

if TRAIN_SCRIPT is None:
    print(f'❌ Could not find training script in:')
    for p in candidates:
        exists = '✓' if p.exists() else '✗'
        print(f'   {exists} {p}')
    print('\n⚠️  Troubleshooting:')
    print('   1. Ensure the clone cell completed successfully')
    print('   2. Check that backend/scripts/train_hybrid.py exists in the repo')
    print('   3. Try re-running the clone cell')
    raise FileNotFoundError(f'Could not locate training script under {REPO_DIR}')

BACKEND_DIR = TRAIN_SCRIPT.parent.parent if 'scripts' in str(TRAIN_SCRIPT) else TRAIN_SCRIPT.parent
REPO_ROOT = REPO_DIR

for p in [str(REPO_ROOT), str(BACKEND_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from backend.feature_extractor import extract_medical_features
    print('✅ import backend.feature_extractor — OK')
except Exception as e:
    try:
        from feature_extractor import extract_medical_features
        print('✅ import feature_extractor — OK (fallback)')
    except Exception:
        print(f'⚠️ Could not import feature_extractor: {e}')

print(f'\nTrain script : {TRAIN_SCRIPT}')
print(f'Repo root    : {REPO_ROOT}')
print(f'Backend dir  : {BACKEND_DIR}')
print(f'sys.path[0]  : {sys.path[0]}')
print('\n✅ Path setup complete!')

In [ ]:
# ── Kaggle dataset diagnostic ────────────────────────────────────────────────
import os
from pathlib import Path

print('=' * 70)
print('📊 DATA DIAGNOSTIC - Kaggle dataset locations')
print('=' * 70)

REPO_ROOT = Path('/kaggle/working/repo')
KAGGLE_INPUT = Path('/kaggle/input')

print('\n1️⃣  Repository contents:')
if REPO_ROOT.exists():
    for item in sorted(REPO_ROOT.iterdir())[:10]:
        marker = '/' if item.is_dir() else ''
        print(f'   {item.name}{marker}')
else:
    print(f'   Repo not found at {REPO_ROOT}')

print('\n2️⃣  /kaggle/input datasets:')
if KAGGLE_INPUT.exists():
    datasets = list(KAGGLE_INPUT.iterdir())
    if datasets:
        for dataset_dir in sorted(datasets):
            if dataset_dir.is_dir():
                print(f'   📁 {dataset_dir.name}/')
                for item in sorted(dataset_dir.iterdir())[:8]:
                    marker = '/' if item.is_dir() else ''
                    print(f'      {item.name}{marker}')
                    # If it's the datasets folder, show subdirs
                    if item.is_dir() and dataset_dir.name == 'datasets':
                        for subitem in sorted(item.iterdir())[:5]:
                            submk = '/' if subitem.is_dir() else ''
                            print(f'         {subitem.name}{submk}')
    else:
        print('   (No datasets mounted)')
else:
    print('   /kaggle/input not found')

print('\n3️⃣  Searching for valid data roots:')
valid_paths = []

# Check all subdirectories under datasets/
if (KAGGLE_INPUT / 'datasets').exists():
    for owner_dir in (KAGGLE_INPUT / 'datasets').iterdir():
        if owner_dir.is_dir():
            for dataset_dir in owner_dir.iterdir():
                if dataset_dir.is_dir():
                    # Check common data root names
                    for candidate in [dataset_dir / 'data', dataset_dir / 'train', dataset_dir]:
                        if candidate.exists():
                            if any((candidate / cls).exists() for cls in ['Normal', 'CIN1', 'CIN2', 'CIN3', 'Cancer', 'train', 'val']):
                                valid_paths.append(candidate)
                                print(f'   ✅ {candidate.relative_to(KAGGLE_INPUT)}')
                                break

# Check direct /kaggle/input paths
for candidate in [
    KAGGLE_INPUT / 'data',
    KAGGLE_INPUT / 'train',
    REPO_ROOT / 'data',
]:
    if candidate.exists() and any((candidate / cls).exists() for cls in ['Normal', 'CIN1', 'CIN2', 'CIN3', 'Cancer', 'train', 'val']):
        valid_paths.append(candidate)
        print(f'   ✅ {candidate}')

if valid_paths:
    print(f'\n✅ Data found at {len(valid_paths)} location(s).')
else:
    print('\n⚠️  Run the dataset mounting cell, then re-run this cell to discover new datasets.')

## Step 3 — Dataset Setup (5-Class Classification)

The model classifies cervical cells into **5 severity grades**:

```
data/
├── train/
│   ├── Normal/        # Healthy cervical tissue
│   ├── CIN1/          # Low-grade dysplasia (Grade 1)
│   ├── CIN2/          # High-grade dysplasia (Grade 2)
│   ├── CIN3/          # High-grade dysplasia (Grade 3)
│   └── Cancer/        # Invasive cervical cancer
└── val/
    ├── Normal/
    ├── CIN1/
    ├── CIN2/
    ├── CIN3/
    └── Cancer/
```

Or a **flat layout** `data/<ClassName>/*.jpg` (auto-split 80/20).  
Synthetic images should have `syn_` in their filename to enable the is_synthetic feature flag.

**5-Class Severity Order:**
- **Normal**: Healthy tissue, no abnormalities
- **CIN1**: Low-grade dysplasia (mild abnormalities)
- **CIN2**: High-grade dysplasia (moderate abnormalities)  
- **CIN3**: High-grade dysplasia (severe abnormalities)
- **Cancer**: Invasive cervical cancer

In [ ]:
# ── Merge map: your folder names → model class names ─────────────────────────
# NOW SUPPORTING 5 CLASSES: Normal, CIN1, CIN2, CIN3, Cancer (no merging)
FOLDER_TO_CLASS = {
    'Normal' : 'Normal',
    'CIN1'   : 'CIN1',
    'CIN2'   : 'CIN2',      # 🟢 Changed: CIN2 stays separate (was HighGrade)
    'CIN3'   : 'CIN3',      # 🟢 Changed: CIN3 stays separate (was HighGrade)
    'Cancer' : 'Cancer',
}
SEVERITY_ORDER = ['Normal', 'CIN1', 'CIN2', 'CIN3', 'Cancer']  # 🟢 5 classes

## Step 4 — Training Configuration

Adjust hyperparameters below before launching training.  
Defaults are tuned for a T4/V100 GPU with the full SIPaKMeD + Herlev dataset.


In [ ]:
# ── Training hyperparameters for Kaggle ───────────────────────────────────
EPOCHS        = 100
BATCH_SIZE    = 16
IMAGE_SIZE    = 384
LEARNING_RATE = 5e-5
WEIGHT_DECAY  = 1e-4
WARMUP_EPOCHS = 5
PATIENCE      = 12
NUM_WORKERS   = 2
SEED          = 42

# Backbone / augmentation options
BACKBONE         = 'auto'   # auto, convnextv2_large, efficientnetv2_l, swinv2_large, maxvit_large
SEARCH_BACKBONES = True
MIXUP_ALPHA      = 0.4
CUTMIX_ALPHA     = 1.0
USE_MIXUP        = True
USE_CUTMIX       = True
USE_SAMPLER      = True
USE_AMP          = True

# Enable stronger augmentations (RandAugment-style)
STRONG_AUG = True

# Kaggle-only data and output directories
KAGGLE_INPUT = Path('/kaggle/input')

# Build candidate list dynamically by searching all datasets
DATA_CANDIDATES = []

# First, check all subdirectories under /kaggle/input/datasets
if (KAGGLE_INPUT / 'datasets').exists():
    for owner_dir in (KAGGLE_INPUT / 'datasets').iterdir():
        if owner_dir.is_dir():
            for dataset_dir in owner_dir.iterdir():
                if dataset_dir.is_dir():
                    # Add both the dataset root and common subdirs
                    for candidate in [dataset_dir / 'data', dataset_dir]:
                        if candidate not in DATA_CANDIDATES:
                            DATA_CANDIDATES.append(candidate)

# Then check direct /kaggle/input paths
for path in [KAGGLE_INPUT / 'data', KAGGLE_INPUT / 'train', REPO_ROOT / 'data']:
    if path not in DATA_CANDIDATES:
        DATA_CANDIDATES.append(path)

# Find the first valid dataset
DATA_DIR = None
for candidate in DATA_CANDIDATES:
    if candidate.exists():
        # Check for train/val split structure
        if (candidate / 'train').exists() and (candidate / 'val').exists():
            DATA_DIR = candidate
            print(f'✅ Found data (train/val split): {candidate}')
            break
        # Check for flat class structure (Normal, CIN1, CIN2, CIN3, Cancer)
        if any((candidate / cls).exists() for cls in ['Normal', 'CIN1', 'CIN2', 'CIN3', 'Cancer']):
            DATA_DIR = candidate
            print(f'✅ Found data (flat classes): {candidate}')
            break

if DATA_DIR is None:
    print('\n❌ Could not find the Kaggle dataset.')
    print('\nSearched locations:')
    for path in DATA_CANDIDATES[:10]:
        exists = '✓' if path.exists() else '✗'
        print(f'   {exists} {path}')
    if len(DATA_CANDIDATES) > 10:
        print(f'   ... and {len(DATA_CANDIDATES) - 10} more')
    print('\n💡 Make sure you mounted the dataset before running this cell.')
    print('   Then re-run the diagnostic cell (Step 3) to see available datasets.')
    raise FileNotFoundError('No valid training dataset found in /kaggle/input')

CHECKPOINT_DIR = WORK_BASE / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print('\n' + '=' * 70)
print('TRAINING CONFIGURATION  (Hybrid Model)')
print('=' * 70)
print(f'  Data dir        : {DATA_DIR}')
print(f'  Checkpoint dir  : {CHECKPOINT_DIR}')
print(f'  Epochs          : {EPOCHS}')
print(f'  Batch size      : {BATCH_SIZE}')
print(f'  Image size      : {IMAGE_SIZE}')
print(f'  LR              : {LEARNING_RATE}')
print(f'  Weight decay    : {WEIGHT_DECAY}')
print(f'  Warmup epochs   : {WARMUP_EPOCHS}')
print(f'  Patience        : {PATIENCE}')
print(f'  Backbone        : {BACKBONE}')
print('=' * 70)

## Step 5 — Launch Training

Runs `backend/train.py` as a subprocess so the new backbone + Mamba pipeline trains on Colab/GPU.  
The script can auto-search ConvNeXt V2 Large, EfficientNetV2-L, Swin Transformer V2, and MaxViT, then keep the best validation macro F1 checkpoint.


# 📝 OPTIONAL: Notes / configuration reference (do not run)
#
# This notebook now trains the new backbone + Mamba pipeline via `backend/train.py`.
# If you need to debug CUDA issues, you can still set:
#
"""
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb=512'
"""
#
# To reduce VRAM use, lower `BATCH_SIZE` or set `BACKBONE='efficientnetv2_l'`.


In [ ]:
# ── Launch training with train_hybrid.py or backend/train.py ─────────────────
import os
import subprocess
import sys

# ⚠️  CRITICAL: Clear problematic CUDA allocator settings that don't work on Kaggle
if 'PYTORCH_CUDA_ALLOC_CONF' in os.environ:
    print('⚠️  Clearing PYTORCH_CUDA_ALLOC_CONF to avoid compatibility issues...')
    del os.environ['PYTORCH_CUDA_ALLOC_CONF']

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

print('=' * 70)
print('🚀 TRAINING: Hybrid Model')
print('=' * 70)

# Construct training command
cmd = [
    sys.executable, str(TRAIN_SCRIPT),
    '--data-dir', str(DATA_DIR),
    '--checkpoint-dir', str(CHECKPOINT_DIR),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--weight-decay', str(WEIGHT_DECAY),
    '--num-workers', str(NUM_WORKERS),
    '--seed', str(SEED),
]

# If launching the new pipeline backend/train.py, add search/backbone args
if TRAIN_SCRIPT.name == 'train.py':
    if SEARCH_BACKBONES:
        cmd.append('--search-backbones')
    if not USE_MIXUP:
        cmd.append('--no-mixup')
    if not USE_CUTMIX:
        cmd.append('--no-cutmix')
    if not USE_SAMPLER:
        cmd.append('--no-sampler')
    if STRONG_AUG:
        cmd.append('--strong-augment')

# If launching legacy train_hybrid.py, add its supported flags
if TRAIN_SCRIPT.name == 'train_hybrid.py':
    if not USE_CUTMIX:
        cmd.append('--no-cutmix')

print('Training command:')
print('  ' + ' '.join(cmd[:3]) + ' \\')
for part in cmd[3:]:
    print(f'  {part} \\')
print()

print('Training in progress...')
print('-' * 70)

# Run training
result = subprocess.run(cmd, cwd=str(BACKEND_DIR))

print('-' * 70)
if result.returncode == 0:
    print('\n✅ TRAINING COMPLETED SUCCESSFULLY')
    print(f'   Checkpoints saved to: {CHECKPOINT_DIR}')
else:
    print(f'\n⚠️  Training exited with code {result.returncode}')
    print('   Check the output above for error details')

In [ ]:
# ── Check the saved checkpoints from the new trainer ─────────────────────────
print('=' * 70)
print('🔍 CHECKPOINT INSPECTION - Did training complete?')
print('=' * 70)

if CHECKPOINT_DIR.exists():
    checkpoints = sorted(CHECKPOINT_DIR.glob('*.pt'), key=lambda p: p.stat().st_mtime, reverse=True)
    if checkpoints:
        print(f'\n✅ {len(checkpoints)} checkpoint file(s) detected:\n')
        for cp in checkpoints:
            size_mb = cp.stat().st_size / 1024 / 1024
            print(f'   📦 {cp.name:30s} ({size_mb:6.1f} MB)')

        history_file = CHECKPOINT_DIR / 'history.json'
        if history_file.exists():
            print(f'\n✅ Training history found: {history_file.name}')
            import json
            with open(history_file) as f:
                history = json.load(f)

            def _normalize_history(h):
                if isinstance(h, dict):
                    if all(isinstance(k, str) and k.isdigit() for k in h.keys()):
                        return [h[k] for k in sorted(h.keys(), key=int)]
                    if 'train' in h and 'val' in h and isinstance(h['train'], list):
                        return [
                            {'train': h['train'][i], 'val': h['val'][i]}
                            for i in range(len(h['train']))
                        ]
                    if 'history' in h and isinstance(h['history'], list):
                        return h['history']
                    if all(isinstance(v, dict) for v in h.values()):
                        return list(h.values())
                    return [h]

                if isinstance(h, list) and h:
                    first = h[0]
                    if isinstance(first, str):
                        parsed = []
                        for item in h:
                            try:
                                parsed.append(json.loads(item))
                            except Exception:
                                parsed.append(item)
                        if any(isinstance(x, dict) for x in parsed):
                            return _normalize_history(parsed)
                        h = parsed
                        first = h[0]
                    if isinstance(first, list):
                        if len(first) == 1 and isinstance(first[0], dict):
                            return [row[0] for row in h]
                        if len(first) == 2 and isinstance(first[0], dict) and isinstance(first[1], dict):
                            return [{'train': row[0], 'val': row[1]} for row in h]
                    if isinstance(first, dict):
                        if 'train' in first and 'val' in first:
                            return h
                        if {'loss', 'accuracy'} <= set(first.keys()):
                            return [{'train': row, 'val': row} for row in h]
                    return h

                return h

            history = _different_history = _normalize_history(history)
            if isinstance(history, list) and history and isinstance(history[0], dict) and 'train' not in history[0] and 'val' not in history[0] and {'loss', 'accuracy'}.issubset(set(history[0].keys())):
                history = [{'train': row, 'val': row} for row in history]
            if isinstance(history, list) and history and isinstance(history[0], dict):
                best_epoch = max(history, key=lambda x: x['val'].get('f1_macro', float('-inf')))
                print(f"   • Epochs completed: {len(history)}")
                print(f"   • Best val accuracy: {best_epoch['val'].get('accuracy', 0):.4f}")
                print(f"   • Best val macro F1: {best_epoch['val'].get('f1_macro', 0):.4f}")
                print(f"   • Best backbone    : {best_epoch.get('backbone', 'N/A')}")
            else:
                print('   ⚠️ Could not interpret history.json as epoch list/dict format.')
                print('   history type:', type(history))
                if isinstance(history, list):
                    print('   first item type:', type(history[0]))
                    if isinstance(history[0], (str, int, float)):
                        print('   sample first item:', history[0])
        print('\nIf these files exist, training completed successfully.')
    else:
        print(f'\n❌ No checkpoints found in: {CHECKPOINT_DIR}')
else:
    print(f'\n❌ Checkpoint directory not found: {CHECKPOINT_DIR}')

In [ ]:
# ── WORKAROUND: Prevent CUDA cleanup errors in future runs ────────────────────
print("\n" + "=" * 70)
print("🔧 CUDA ERROR FIX for Next Training Run")
print("=" * 70)
print("\nTo prevent 'CUDA event creation' errors on next training:")
print("\n1️⃣  Open Step 4 (Training Configuration)")
print("2️⃣  Add this line BEFORE training command:\n")
print("   os.environ['CUDA_LAUNCH_BLOCKING'] = '1'")
print("   os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb=512'\n")
print("3️⃣  Also consider disabling AMP (mixed precision) if error persists:")
print("   • In Step 5, add: --no-amp\n")
print("Example Step 5 command:")
print("   python train_hybrid.py \\")
print("     --data-dir /kaggle/input/datasets/shubhrawat00/dataset/data \\")
print("     --epochs 180 --batch-size 64 \\")
print("     --no-amp  # ← Add this for stability\n")
print("=" * 70)

## Step 6 — Inspect Saved Checkpoints

In [ ]:
import torch
from pathlib import Path

print(f"📁 Checkpoint directory: {CHECKPOINT_DIR}\n")

# ── List all .pt files ────────────────────────────────────────────────────────
pt_files = sorted(CHECKPOINT_DIR.rglob("*.pt"))
if not pt_files:
    print("⚠️  No .pt checkpoints found. Check that training completed.")
else:
    print(f"{'File':<30} {'Size (MB)':>10}")
    print("-" * 42)
    for p in pt_files:
        size = p.stat().st_size / 1e6
        print(f"  {p.name:<28} {size:>8.1f} MB")

# ── Load best_model.pt and print metadata ─────────────────────────────────────
best_ckpt_path = CHECKPOINT_DIR / "best_model.pt"
swa_ckpt_path  = CHECKPOINT_DIR / "swa_model.pt"

def show_ckpt(path, label):
    if not path.exists():
        print(f"\n  {label}: not found")
        return None
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    print(f"\n{'='*50}")
    print(f"  {label}  ({path.name})")
    print(f"{'='*50}")
    print(f"  Version      : {ckpt.get('version', 'N/A')}")
    print(f"  Epoch        : {ckpt.get('epoch', 'N/A')}")
    print(f"  Classes      : {ckpt.get('class_names', 'N/A')}")
    print(f"  Num classes  : {ckpt.get('num_classes', 'N/A')}")
    print(f"  Num features : {ckpt.get('num_features', 'N/A')}")
    print(f"  Input size   : {ckpt.get('input_size', 'N/A')}")
    print(f"  Val accuracy : {ckpt.get('val_acc', 0):.2f}%")
    print(f"  Balanced acc : {ckpt.get('val_bacc', 0):.2f}%")
    print(f"  Macro F1     : {ckpt.get('macro_f1', 0):.4f}")
    print(f"  SWA          : {ckpt.get('use_swa', False)}")
    print(f"  Severity     : {ckpt.get('severity_order', 'N/A')}")
    return ckpt

best_ckpt = show_ckpt(best_ckpt_path, "BEST MODEL")
swa_ckpt  = show_ckpt(swa_ckpt_path,  "SWA MODEL")


## Step 7 — Plot Training History

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

hist_path = CHECKPOINT_DIR / 'history.json'
if not hist_path.exists():
    print('⚠️  history.json not found. Run training first.')
else:
    with open(hist_path) as f:
        history = json.load(f)

    def _normalize_history(h):
        if isinstance(h, dict):
            if all(isinstance(k, str) and k.isdigit() for k in h.keys()):
                return [h[k] for k in sorted(h.keys(), key=int)]
            if 'train' in h and 'val' in h and isinstance(h['train'], list):
                return [
                    {'train': h['train'][i], 'val': h['val'][i]}
                    for i in range(len(h['train']))
                ]
            if 'history' in h and isinstance(h['history'], list):
                return h['history']
            if all(isinstance(v, dict) for v in h.values()):
                return list(h.values())
            return [h]

        if isinstance(h, list) and h:
            first = h[0]
            if isinstance(first, str):
                parsed = []
                for item in h:
                    try:
                        parsed.append(json.loads(item))
                    except Exception:
                        parsed.append(item)
                h = parsed
                first = h[0]
            if isinstance(first, list) and len(first) == 1 and isinstance(first[0], dict):
                return [row[0] for row in h]
            if isinstance(first, list):
                flattened = []
                for row in h:
                    if isinstance(row, list) and len(row) == 1:
                        flattened.append(row[0])
                    else:
                        flattened.append(row)
                return _normalize_history(flattened)
            if isinstance(first, dict):
                if 'train' in first and 'val' in first:
                    return h
                if {'loss', 'accuracy'} <= set(first.keys()):
                    return [{'train': row, 'val': row} for row in h]
            return h

        return h

    history = _normalize_history(history)
    if isinstance(history, list) and history and not isinstance(history[0], dict):
        print('⚠️  history.json did not normalize to epoch dicts. First item type:', type(history[0]))
        print('   first item repr:', repr(history[0])[:400])

    epochs_ran = len(history)
    xs = list(range(1, epochs_ran + 1))
    tr_loss = [row['train']['loss'] for row in history]
    va_loss = [row['val']['loss'] for row in history]
    tr_acc = [row['train']['accuracy'] for row in history]
    va_acc = [row['val']['accuracy'] for row in history]
    va_f1 = [row['val']['f1_macro'] for row in history]
    va_prec = [row['val']['precision_macro'] for row in history]
    va_rec = [row['val']['recall_macro'] for row in history]

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Training History — Backbone + Mamba Pipeline', fontsize=14, fontweight='bold')

    ax = axes[0, 0]
    ax.plot(xs, tr_loss, label='Train loss', color='#2196F3')
    ax.plot(xs, va_loss, label='Val loss', color='#F44336')
    ax.set_title('Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(alpha=0.3)

    ax = axes[0, 1]
    ax.plot(xs, tr_acc, label='Train acc', color='#2196F3')
    ax.plot(xs, va_acc, label='Val acc', color='#F44336')
    ax.set_title('Accuracy')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(alpha=0.3)

    ax = axes[1, 0]
    ax.plot(xs, va_f1, label='Val macro F1', color='#4CAF50', linewidth=2)
    ax.plot(xs, va_prec, label='Val precision', color='#FF9800', linestyle='--')
    ax.plot(xs, va_rec, label='Val recall', color='#9C27B0', linestyle=':')
    ax.set_title('Validation Metrics')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Score')
    ax.legend()
    ax.grid(alpha=0.3)

    ax = axes[1, 1]
    best_epoch = int(np.argmax(va_f1)) + 1
    ax.plot(xs, va_f1, color='#4CAF50', linewidth=2)
    ax.axvline(best_epoch, color='gray', linestyle=':', label=f'Best epoch={best_epoch}')
    ax.set_title('Validation Macro F1')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('F1')
    ax.legend()
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(CHECKPOINT_DIR / 'training_history.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\n  Best epoch   : {best_epoch}  (Macro F1 = {max(va_f1):.4f})')
    print(f"  Plot saved to: {CHECKPOINT_DIR / 'training_history.png'}")

## Step 8 — Inference on a Sample Image

Loads the best checkpoint and runs TTA-based inference with the new model pipeline.

In [ ]:
import json
from pathlib import Path

try:
    from backend.inference import predict_image
except ImportError:
    from inference import predict_image

MODEL_PATH = CHECKPOINT_DIR / 'best_model.pt'
if not MODEL_PATH.exists():
    MODEL_PATH = CHECKPOINT_DIR / 'swa_model.pt'

if not MODEL_PATH.exists():
    print('⚠️  No trained model checkpoint found. Run training first.')
else:
    print(f'✅ Using checkpoint: {MODEL_PATH}')

    INFERENCE_IMAGE = None   # Set to a path if you want to test a specific image

    if INFERENCE_IMAGE is None and (DATA_DIR / 'val').exists():
        for cls_dir in sorted((DATA_DIR / 'val').iterdir()):
            if cls_dir.is_dir():
                imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')) + list(cls_dir.glob('*.jpeg'))
                if imgs:
                    INFERENCE_IMAGE = str(imgs[0])
                    true_label = cls_dir.name
                    break
    elif INFERENCE_IMAGE is not None:
        true_label = Path(INFERENCE_IMAGE).parent.name

    if INFERENCE_IMAGE is None:
        print('No validation images found for inference.')
    else:
        result = predict_image(INFERENCE_IMAGE, [MODEL_PATH], image_size=IMAGE_SIZE, tta_views=6, device='cuda' if torch.cuda.is_available() else 'cpu')
        print(f"\nImage         : {INFERENCE_IMAGE}")
        print(f"True label    : {true_label}")
        print(f"Predicted     : {result['predicted_class']}  (conf: {result['confidence']*100:.1f}%)")
        print('\nClass probabilities:')
        for cls_name, prob in result['probabilities'].items():
            print(f'  {cls_name:<12} {prob*100:5.1f}%')

## Step 9 — Package & Download Checkpoints

In [ ]:
import json
from pathlib import Path

metrics_path = CHECKPOINT_DIR / 'metrics.json'
print(f'Checking for metrics.json at: {metrics_path}')
print(f'Checkpoint dir exists: {CHECKPOINT_DIR.exists()}')
if CHECKPOINT_DIR.exists():
    print('Checkpoint directory contents:')
    for path in sorted(CHECKPOINT_DIR.iterdir()):
        print(f'  - {path.name}')
if not metrics_path.exists():
    print('⚠️  metrics.json not found. Run training first.')
else:
    with open(metrics_path) as f:
        metrics = json.load(f)

    print('=' * 70)
    print('MODEL PERFORMANCE ANALYSIS')
    print('=' * 70)
    print(f"\nAccuracy         : {metrics.get('accuracy', 0):.4f}")
    print(f"Precision (macro): {metrics.get('precision_macro', 0):.4f}")
    print(f"Recall (macro)   : {metrics.get('recall_macro', 0):.4f}")
    print(f"F1-score (macro) : {metrics.get('f1_macro', 0):.4f}")
    print(f"ROC-AUC (ovr)    : {metrics.get('roc_auc_ovr_macro', 'N/A')}")
    print('')
    print('Confusion matrix saved to:')
    print(f'  {CHECKPOINT_DIR / "confusion_matrix.csv"}')
    print('Classification report saved to:')
    print(f'  {CHECKPOINT_DIR / "classification_report.json"}')

    cm = metrics.get('confusion_matrix', [])
    if cm:
        print('\nConfusion matrix:')
        for row in cm:
            print('  ' + ' '.join(f'{int(v):4d}' for v in row))

In [ ]:
print('=' * 70)
print('BACKBONE SEARCH / TUNING NOTES')
print('=' * 70)

print('If you want to retrain with a different configuration, edit Step 4:')
print('  - BACKBONE = auto / convnextv2_large / efficientnetv2_l / swinv2_large / maxvit_large')
print('  - SEARCH_BACKBONES = True to benchmark all supported backbones')
print('  - IMAGE_SIZE = 384 or 448')
print('  - EPOCHS = 80 to 100')
print('  - BATCH_SIZE = 16 or lower if VRAM is limited')
print('  - LEARNING_RATE = 3e-5 to 5e-5')
print('  - USE_MIXUP / USE_CUTMIX / USE_SAMPLER as needed')

print('\nRecommended next experiment order:')
print('  1. Run auto backbone search with IMAGE_SIZE=384')
print('  2. Re-run the best backbone at IMAGE_SIZE=448 if VRAM allows')
print('  3. Enlarge MixUp/CutMix if validation overfits')
print('  4. Keep the checkpoint with the best macro F1')
print('=' * 70)

## Step 11 — Retrain with Optimized Settings

Recommended next pass:
- `BACKBONE='auto'` and `SEARCH_BACKBONES=True`
- `IMAGE_SIZE=384` for the first benchmark run
- `EPOCHS=80` to `100`
- `BATCH_SIZE=16` or lower if VRAM is tight
- keep `USE_AMP=True` on GPU

## Step 10 — Backbone Search / Tuning Notes

Use these settings to push validation macro F1 higher:
- Try `BACKBONE='auto'` with `SEARCH_BACKBONES=True` first.
- If VRAM is limited, prefer `efficientnetv2_l`.
- If memory permits, try `IMAGE_SIZE=448` for the best backbone.
- Keep MixUp, CutMix, and the weighted sampler enabled unless you are debugging.

In [ ]:
import zipfile
from pathlib import Path

zip_path = WORK_BASE / 'trained_model_backbone_mamba.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(CHECKPOINT_DIR.rglob('*')):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(CHECKPOINT_DIR))

size_mb = zip_path.stat().st_size / 1e6
print(f'✅ Zip created: {zip_path}  ({size_mb:.1f} MB)')
print('Contents:')
with zipfile.ZipFile(zip_path) as zf:
    for name in zf.namelist():
        info = zf.getinfo(name)
        print(f'  {name:<35} {info.file_size/1e6:.1f} MB')

try:
    from google.colab import files
    files.download(str(zip_path))
    print('\n✅ Download started (Colab).')
except ImportError:
    print(f'\nℹ️  Colab not detected. Output zip is here: {zip_path}')